# 90 · Gate A report

Reads **only** `runs/` (every `metrics.json`, `DONE`, `FAILED`) and produces the tables and figures of the technical
report plus `report/commercial_summary.md`. Nothing is trained here, so it is cheap to rerun after any stage finishes.

Sections: unit status · main results (test AUC, PR-AUC, capture@10% by tier, mean ± sd over seeds) · paired lifts
over XGBoost · DGraph-Fin against the published range · Tier 1 vs Tier 3 · LUPI students against both controls ·
calibration, stability, business · the pre-registered criteria, checked.

Pre-registered criteria (from the plan, fixed before the first run):

- A model family is called **lift** only if its AUC *and* PR-AUC gain over XGBoost has a 95% CI excluding zero on at least
  two datasets, and it does not lose calibration (isotonic ECE not worse by > 0.005) or stability (PSI not worse by > 0.05).
- The hybrid (Tier 3) is the primary commercial candidate.
- Tier 1 vs Tier 3 is the key commercial comparison.
- LUPI: a student is said to have received graph knowledge only if its lift beats **both** the self-distillation and the
  random-graph control.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 90_report dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/90_report/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "all"
seed = 0
graph_builder = "all"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results
from IPython.display import display

STAGE = "90_report"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

## Unit status

In [ ]:
from src.eval import results as R

status = R.unit_status()
pivot = status.groupby(["stage", "status"]).size().unstack(fill_value=0)
display(pivot)
not_done = status[status.status != "done"]
if len(not_done):
    print(f"{len(not_done)} units not done (results below are partial):")
    display(not_done.head(40))

## Main table: test metrics by tier, mean ± sd over seeds

In [ ]:
df = R.collect()
assert len(df), "no metrics.json found under runs/"
df["tier"] = df["stage"].str[:2].map({"02": "0 tabular", "04": "1 graph features", "05": "2 gnn", "06": "3 hybrid", "07": "4 lupi student"})
main = df[(df.split == "test") & df.tier.notna() & df.metric.isin(["auc", "pr_auc", "capture_10", "brier", "ece"])]
agg = main.groupby(["dataset", "tier", "graph_builder", "model", "metric"])["value"].agg(["mean", "std", "count"]).reset_index()
wide = agg.pivot_table(index=["dataset", "tier", "graph_builder", "model"], columns="metric", values="mean")
wide_sd = agg.pivot_table(index=["dataset", "tier", "graph_builder", "model"], columns="metric", values="std")
n_seeds = agg.pivot_table(index=["dataset", "tier", "graph_builder", "model"], columns="metric", values="count")["auc"]
fmt = wide.copy().astype(object)
for c in wide.columns:
    fmt[c] = [R.fmt_mean_std(m, s) for m, s in zip(wide[c], wide_sd[c])]
fmt["seeds"] = n_seeds.astype(int)
(paths.REPORT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
wide.to_csv(paths.REPORT_ROOT / "results_test_mean.csv"); wide_sd.to_csv(paths.REPORT_ROOT / "results_test_sd.csv")
fmt

In [ ]:
for ds, g in wide.reset_index().groupby("dataset"):
    g = g.sort_values("auc", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(g) + 1.2))
    labels = g["tier"].str[0] + " | " + g["graph_builder"] + " | " + g["model"]
    sd = wide_sd.reset_index().set_index(["tier", "graph_builder", "model"]).loc[list(zip(g.tier, g.graph_builder, g.model)), "auc"].fillna(0).to_numpy()
    ax.barh(labels, g["auc"], xerr=sd, color=["C" + t[0] for t in g["tier"]])
    ax.invert_yaxis(); ax.set(title=f"{ds}: test AUC (mean ± sd over seeds)", xlim=(max(0.5, g["auc"].min() - 0.05), min(1.0, g["auc"].max() + 0.02)))
    fig.tight_layout(); fig.savefig(paths.REPORT_ROOT / "figures" / f"auc_{ds}.png", dpi=110); plt.show()

## Paired lifts over Tier 0 XGBoost

Each unit stored a paired-bootstrap comparison against the stage-02 XGBoost of the same seed. Here: mean difference over seeds, the mean CI bounds, and in how many seeds the CI excluded zero.

In [ ]:
cmp = R.collect_compare()
cmp = cmp[cmp.get("baseline", "xgb").eq("xgb") if "baseline" in cmp.columns else slice(None)] if len(cmp) else cmp
cmp = cmp[cmp.stage.isin(["04_graph_features_xgb", "05_gnn", "06_hybrid"])] if len(cmp) else cmp
lift = pd.DataFrame()
if len(cmp):
    lift = cmp.groupby(["dataset", "stage", "graph_builder", "model", "metric"]).agg(
        diff=("diff", "mean"), ci_low=("ci_low", "mean"), ci_high=("ci_high", "mean"),
        n_sig=("significant", "sum"), n_seeds=("significant", "size")).reset_index()
    lift["sig_seeds"] = lift["n_sig"].astype(int).astype(str) + "/" + lift["n_seeds"].astype(str)
    for ds, g in lift[lift.metric == "auc"].groupby("dataset"):
        g = g.assign(label=g.stage.str[:2] + " | " + g.graph_builder + " | " + g.model)
        fig = viz.plot_intervals(g, label="label", title=f"{ds}: test AUC minus XGBoost (mean over seeds, mean 95% CI)")
        fig.savefig(paths.REPORT_ROOT / "figures" / f"lift_{ds}.png", dpi=110); plt.show()
    lift.to_csv(paths.REPORT_ROOT / "lifts_vs_xgb.csv", index=False)
lift.round(4) if len(lift) else print("no comparisons yet")

## DGraph-Fin against the published range

In [ ]:
ref = config.load_config("dgraphfin").get("reference_auc", {})
dg = wide.reset_index()
dg = dg[dg.dataset == "dgraphfin"][["tier", "graph_builder", "model", "auc"]].copy()
dg["paper_auc"] = dg["model"].map(ref)
dg["gap"] = dg["auc"] - dg["paper_auc"]
dg.round(4) if len(dg) else print("no dgraphfin results yet")

## Tier 1 vs Tier 3: does the lift need a neural network?

In [ ]:
t13 = lift[(lift.metric == "auc") & lift.stage.isin(["04_graph_features_xgb", "06_hybrid"])] if len(lift) else pd.DataFrame()
if len(t13):
    piv = t13.pivot_table(index=["dataset", "graph_builder"], columns="stage", values="diff")
    piv.columns = [c[:2] + "_" + ("tier1" if c.startswith("04") else "tier3") for c in piv.columns]
    if piv.shape[1] == 2:
        piv["tier3_minus_tier1"] = piv.iloc[:, 1] - piv.iloc[:, 0]
    display(piv.round(4))
else:
    print("need 04 and 06 results")

## LUPI: students against both controls

For each dataset and seed, the selected student lift (over plain XGBoost) from the *structured* teacher (native / relational / shared_attribute / kNN) is compared with the same-seed lifts from the self-distillation teacher and from the random-control teacher.

In [ ]:
lupi_rows = []
for mpath in paths.RUNS_ROOT.glob("07_lupi_distill/*/*/*/metrics.json"):
    unit = paths.parse_run_dir(mpath.parent)
    payload = json.loads(mpath.read_text())
    plain_auc = payload["models"]["xgb_plain"]["test"]["auc"]
    plain_pr = payload["models"]["xgb_plain"]["test"]["pr_auc"]
    for tn, s in payload.get("lupi", {}).get("selected", {}).items():
        m = payload["models"].get(f"student_{tn}", {}).get("test", {})
        cmp_r = [c for c in payload.get("compare", []) if c["model"] == f"student_{tn}" and c["metric"] == "auc"]
        lupi_rows.append({**unit, "teacher": tn, "lam": s["lam"], "T": s["T"], "plain_auc": plain_auc, "student_auc": m.get("auc"),
                          "lift_auc": (m.get("auc") or np.nan) - plain_auc, "lift_pr_auc": (m.get("pr_auc") or np.nan) - plain_pr,
                          "ci_low": cmp_r[0]["ci_low"] if cmp_r else np.nan, "ci_high": cmp_r[0]["ci_high"] if cmp_r else np.nan})
lupi_df = pd.DataFrame(lupi_rows)
if len(lupi_df):
    lupi_df["arm"] = np.where(lupi_df.teacher == "self_xgb", "self_distill", np.where(lupi_df.graph_builder == "random_control", "random_control", "structured"))
    # keep the self-distillation control once per dataset/seed (it is identical across builder units)
    lupi_df = lupi_df[(lupi_df.arm != "self_distill") | (lupi_df.graph_builder == lupi_df.groupby(["dataset", "seed"]).graph_builder.transform("first"))]
    summ = lupi_df.groupby(["dataset", "arm", "teacher"]).agg(lift_auc=("lift_auc", "mean"), lift_auc_sd=("lift_auc", "std"),
                                                              lift_pr_auc=("lift_pr_auc", "mean"), ci_low=("ci_low", "mean"), ci_high=("ci_high", "mean"),
                                                              seeds=("seed", "nunique"), lam=("lam", "median"), T=("T", "median")).reset_index()
    display(summ.round(4))
    for ds, g in summ.groupby("dataset"):
        fig = viz.plot_intervals(g.assign(label=g.arm + " | " + g.teacher), value="lift_auc", label="label",
                                 title=f"{ds}: LUPI student AUC lift over plain XGBoost")
        fig.savefig(paths.REPORT_ROOT / "figures" / f"lupi_{ds}.png", dpi=110); plt.show()
    summ.to_csv(paths.REPORT_ROOT / "lupi_summary.csv", index=False)
else:
    summ = pd.DataFrame(); print("no LUPI results yet")

## Calibration, stability, business (from stage 08)

In [ ]:
cal_rows, biz_rows = [], []
for mpath in paths.RUNS_ROOT.glob("08_calibration_stability_business/*/*/*/metrics.json"):
    unit = paths.parse_run_dir(mpath.parent)
    payload = json.loads(mpath.read_text())
    for label, c in payload.get("calibration", {}).items():
        cal_rows.append({**unit, "model": label, **c, "psi": payload.get("stability", {}).get("psi_val_to_test", {}).get(label)})
    for r in payload.get("business", {}).get("table", []):
        biz_rows.append({**unit, **r, "target_bad_rate": payload["business"]["target_bad_rate"]})
cal = pd.DataFrame(cal_rows); biz = pd.DataFrame(biz_rows)
if len(cal):
    cal_s = cal.groupby(["dataset", "model"]).agg(auc=("auc", "mean"), ece_raw=("ece_raw", "mean"), ece_isotonic=("ece_isotonic", "mean"),
                                                  brier_isotonic=("brier_isotonic", "mean"), psi=("psi", "mean"), seeds=("seed", "nunique")).reset_index()
    display(cal_s.sort_values(["dataset", "auc"], ascending=[True, False]).round(4)); cal_s.to_csv(paths.REPORT_ROOT / "calibration_stability.csv", index=False)
else:
    cal_s = pd.DataFrame(); print("no stage 08 results yet")
if len(biz):
    biz_s = biz.groupby(["dataset", "model"]).agg(approval_at_matched_bad_rate=("approval_at_matched_bad_rate", "mean"),
                                                  approval_delta_pp=("approval_delta_pp", "mean"), bad_rate_delta_pp=("bad_rate_delta_pp", "mean"),
                                                  capture_10=("capture_10", "mean"), target_bad_rate=("target_bad_rate", "mean"), seeds=("seed", "nunique")).reset_index()
    display(biz_s.sort_values(["dataset", "approval_delta_pp"], ascending=[True, False]).round(4)); biz_s.to_csv(paths.REPORT_ROOT / "business.csv", index=False)
else:
    biz_s = pd.DataFrame()

## Pre-registered criteria, checked

In [ ]:
criteria = []


def family_lift(stage_prefix, name):
    """Lift = AUC and PR-AUC CI excludes zero on >= 2 datasets (majority of seeds), no calibration/stability loss."""
    if not len(lift):
        return {"criterion": name, "status": "insufficient data", "detail": "no comparisons"}
    g = lift[lift.stage.str.startswith(stage_prefix)]
    ok_ds = []
    for ds, gg in g.groupby("dataset"):
        best = gg[gg.metric == "auc"].sort_values("diff", ascending=False).head(1)
        if best.empty:
            continue
        key = (best.graph_builder.iloc[0], best.model.iloc[0])
        sel = gg[(gg.graph_builder == key[0]) & (gg.model == key[1])]
        auc_ok = (sel[sel.metric == "auc"].n_sig >= np.ceil(sel[sel.metric == "auc"].n_seeds / 2)).all() and (sel[sel.metric == "auc"].ci_low > 0).all()
        pr_ok = (sel[sel.metric == "pr_auc"].ci_low > 0).all() if (sel.metric == "pr_auc").any() else False
        cal_ok = True
        if len(cal_s):
            b = cal_s[(cal_s.dataset == ds) & (cal_s.model == "02:xgb")]
            c = cal_s[(cal_s.dataset == ds) & cal_s.model.str.contains(f"{stage_prefix}:.*{key[1]}", regex=True)]
            if len(b) and len(c):
                cal_ok = (c.ece_isotonic.iloc[0] - b.ece_isotonic.iloc[0] <= 0.005) and ((c.psi.iloc[0] or 0) - (b.psi.iloc[0] or 0) <= 0.05)
        if auc_ok and pr_ok and cal_ok:
            ok_ds.append(f"{ds} ({key[0]}/{key[1]})")
    return {"criterion": name, "status": "PASS" if len(ok_ds) >= 2 else "FAIL", "detail": f"datasets with lift: {ok_ds}"}


criteria.append(family_lift("04", "Tier 1 graph features = lift"))
criteria.append(family_lift("05", "Tier 2 pure GNN = lift (not expected)"))
criteria.append(family_lift("06", "Tier 3 hybrid = lift (primary candidate)"))

if len(summ):
    passed = []
    for ds, g in summ.groupby("dataset"):
        s = g[g.arm == "structured"].lift_auc.max() if (g.arm == "structured").any() else np.nan
        selfd = g[g.arm == "self_distill"].lift_auc.max() if (g.arm == "self_distill").any() else np.nan
        rc = g[g.arm == "random_control"].lift_auc.max() if (g.arm == "random_control").any() else np.nan
        ci_ok = (g[g.arm == "structured"].ci_low > 0).any()
        if np.isfinite(s) and ci_ok and s > np.nan_to_num(selfd, nan=-1) and s > np.nan_to_num(rc, nan=-1):
            passed.append(f"{ds}: structured {s:.4f} > self {selfd:.4f}, random {rc:.4f}")
    criteria.append({"criterion": "LUPI student received graph knowledge (beats both controls, CI > 0)",
                     "status": "PASS" if passed else "FAIL", "detail": passed or "no dataset where structured teacher beats both controls"})
else:
    criteria.append({"criterion": "LUPI student received graph knowledge", "status": "insufficient data", "detail": ""})

if len(dg) and dg.paper_auc.notna().any():
    within = dg.dropna(subset=["paper_auc"]).assign(ok=lambda d: d.gap.abs() <= 0.02)
    criteria.append({"criterion": "DGraph-Fin reproduces published AUC within 0.02", "status": "PASS" if within.ok.all() else "FAIL",
                     "detail": within[["model", "auc", "paper_auc"]].round(4).to_dict(orient="records")})
crit_df = pd.DataFrame(criteria)
crit_df.to_csv(paths.REPORT_ROOT / "criteria.csv", index=False)
crit_df

## Commercial summary (two pages, generated)

In [ ]:
def _fmt(x, d=4):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{d}f}"


lines = ["# Gate A: graph signal for credit default, commercial summary", "",
         f"_Generated {time.strftime('%Y-%m-%d %H:%M')} from {int((status.status == 'done').sum())}/{len(status)} finished units._", "",
         "## What was tested", "",
         "Four public credit datasets (Taiwan card default, Home Credit, Lending Club, DGraph-Fin). Tier 0 XGBoost is the reference. "
         "Challengers: hand-built graph features (Tier 1), GNNs (Tier 2), GNN-embedding + XGBoost hybrid (Tier 3), and a tabular student "
         "distilled from a graph teacher (Tier 4, LUPI). Every number is on a frozen test split, five seeds, paired bootstrap 95% CI against XGBoost.", "",
         "## Headline numbers (test AUC, mean over seeds)", ""]
for ds, g in wide.reset_index().groupby("dataset"):
    g = g.sort_values("auc", ascending=False)
    base = g[(g.tier == "0 tabular") & (g.model == "xgb")]
    lines.append(f"**{ds}** — XGBoost {_fmt(base.auc.iloc[0]) if len(base) else 'n/a'}; best challenger "
                 f"{g.iloc[0].tier} / {g.iloc[0].graph_builder} / {g.iloc[0].model} at {_fmt(g.iloc[0].auc)}.")
lines += ["", "## Lift over XGBoost (paired 95% CI; 'sig' = seeds where CI excludes zero)", ""]
if len(lift):
    for _, r in lift[lift.metric == "auc"].sort_values(["dataset", "diff"], ascending=[True, False]).groupby("dataset").head(3).iterrows():
        lines.append(f"- {r.dataset}: {r.stage[:2]} {r.graph_builder}/{r.model}: ΔAUC {r['diff']:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}], sig {r.sig_seeds}")
lines += ["", "## Business translation (stage 08, mean over seeds)", ""]
if len(biz_s):
    for _, r in biz_s[biz_s.model != "baseline"].sort_values(["dataset", "approval_delta_pp"], ascending=[True, False]).groupby("dataset").head(2).iterrows():
        lines.append(f"- {r.dataset}: {r.model} approves {r.approval_delta_pp:+.2f} pp more applicants at XGBoost's bad rate ({r.target_bad_rate:.3%} at 80% approval); capture@10% {r.capture_10:.3f}")
lines += ["", "## Learning using privileged information", ""]
if len(summ):
    for _, r in summ.sort_values(["dataset", "lift_auc"], ascending=[True, False]).iterrows():
        lines.append(f"- {r.dataset}: {r.arm} teacher {r.teacher}: student ΔAUC {r.lift_auc:+.4f} (λ={r.lam}, T={r['T']}, {int(r.seeds)} seeds)")
lines += ["", "## Pre-registered criteria", ""]
for _, r in crit_df.iterrows():
    lines.append(f"- **{r.status}** — {r.criterion}. {r.detail}")
lines += ["", "## What this does and does not show", "",
          "- All labels are public proxies for default; no number here is a calibrated PD for a real portfolio.",
          "- On three datasets the graph is constructed by us and is framed as a structural regulariser, not as evidence of real borrower relationships. "
          "DGraph-Fin is the only dataset with real edges.",
          "- The hybrid and the distilled student are the only shapes that could go on a serving path; pure GNNs are evidence, not candidates.",
          "- Figures: `report/figures/`. Tables: `report/*.csv`. Full detail: the executed notebooks under `runs/`."]
out = paths.REPORT_ROOT / "commercial_summary.md"
out.write_text("\n".join(lines), encoding="utf-8")
print(f"wrote {out} ({len(lines)} lines)")
print("\n".join(lines[:40]))

In [ ]:
summary = {"units_done": int((status.status == "done").sum()), "units_total": int(len(status)),
           "criteria": crit_df.set_index("criterion")["status"].to_dict(), "elapsed_s": round(time.time() - t0, 1)}
results.save_metrics(run_dir, {"models": {}, "report": summary})
ckpt.mark_done(run_dir, summary)
summary